<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Build a Two-Agent System</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Build a simple **two-agent system** with clear role separation and basic message passing.

**Exercise Task: Budget Research Assistant**

Create a two-agent system that helps plan a budget trip from **Riyadh to Istanbul**:

- **Planner Agent**: Breaks down the task and creates a clear plan
- **Executor Agent**: Carries out the research and calculations

**Requirements:**
- One agent responsible for planning
- One agent responsible for executing tasks (research + math)
- Simple message passing between the two agents
- Log the communication between agents

---

## 1. Setup

In [1]:
!pip install -q --force-reinstall "langchain>=0.1.0,<1.0" "langchain-core==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" langsmith duckduckgo-search ddgs faiss-cpu beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.

In [2]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

Enter your OpenAI API key: ··········
Setup complete!


In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
from langsmith import Client

client = Client()

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
react_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

tools = [search_tool, calculate_math]

agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Tools, LLM, and Agent ready")

Tools, LLM, and Agent ready


## TODO: Define Role-Specific Instructions

In [4]:
# TODO 1: Create role-specific instructions

planner_instruction = """You are a Planner Agent. Your only job is to break down the user's task into clear, logical, numbered steps.
Do not perform research or calculations yourself. Just create a good plan."""

researcher_instruction = """You are an Executor Agent. You are responsible for both research (using Web_Search)
and performing calculations (using calculate_math). Follow the plan given by the Planner and produce a clean final summary."""

print("Role instructions ready (fill in the TODOs)")

Role instructions ready (fill in the TODOs)


## TODO: Create the Two Agents

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Planner Agent - Simple LLM chain (no tools)
planner_prompt = ChatPromptTemplate.from_template(
    """You are a Planner Agent. Your ONLY job is to break down the user's request into 3-4 clear, focused steps.

Task: {input}

Create a short, numbered plan that the Executor can actually perform using web search and math calculations.
Focus only on research and calculation steps. Do not include booking, packing, or visa steps."""
)

planner_chain = planner_prompt | llm

# Executor Agent
executor_instruction = """You are an Executor Agent. Follow the plan from the Planner exactly.

Use the Web_Search tool for research and the calculate_math tool for all calculations.
Extract numbers carefully from research results.
Give a clean, professional final summary with all numbers in SAR."""

executor_prompt = react_prompt.partial(system_message=executor_instruction)

executor_agent = create_react_agent(
    llm=llm,
    tools=[search_tool, calculate_math],
    prompt=executor_prompt
)

executor_executor = AgentExecutor(
    agent=executor_agent,
    tools=[search_tool, calculate_math],
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Planner and Executor Agents created!")

Planner and Executor Agents created!


## TODO: Implement Simple Message Passing

In [7]:
def run_two_agent_system(user_query: str):
    print("Starting Two-Agent Budget Assistant\n")
    interactions = []

    # Step 1: Planner Agent
    print("Planner Agent: Creating plan...")
    plan_input = f"Create a short, focused step-by-step plan for this task: {user_query}"

    plan_result = planner_chain.invoke({"input": plan_input})
    plan_text = plan_result.content if hasattr(plan_result, "content") else str(plan_result)

    interactions.append({"from": "Planner", "to": "Executor", "message": plan_text})
    print(plan_text + "\n")

    # Step 2: Executor Agent
    print("Executor Agent: Executing the plan...")
    executor_input = f"""Follow this plan from the Planner:

{plan_text}

Original task: {user_query}

Perform the required research using Web_Search and all calculations using calculate_math.
Provide a clean final summary with all numbers in SAR."""

    executor_result = executor_executor.invoke({"input": executor_input})

    interactions.append({"from": "Executor", "to": "User", "message": "Task completed"})

    # Final Output
    print("\n" + "="*70)
    print("FINAL ANSWER:")
    print(executor_result["output"])

    print("\n" + "="*70)
    print("AGENT COMMUNICATION LOG:")
    for i, msg in enumerate(interactions, 1):
        print(f"{i}. {msg['from']} → {msg['to']}")

    return executor_result["output"]

# Run the system
result = run_two_agent_system("Plan a budget trip from Riyadh to Istanbul for 7 days")

Starting Two-Agent Budget Assistant

Planner Agent: Creating plan...
1. **Research Flight Options**: Use flight comparison websites (e.g., Skyscanner, Google Flights) to find the most affordable round-trip flights from Riyadh to Istanbul. Note the prices and travel times for different airlines.

2. **Estimate Accommodation Costs**: Search for budget accommodation options in Istanbul (e.g., hostels, budget hotels, or Airbnb) for 7 nights. Compare prices and read reviews to find the best value.

3. **Calculate Daily Expenses**: Research the average daily costs for food, transportation, and activities in Istanbul. Create a budget estimate for each category and multiply by 7 days to get a total.

4. **Total Budget Calculation**: Add the costs from steps 1, 2, and 3 to determine the overall budget for the trip, ensuring to include a buffer for unexpected expenses.

Executor Agent: Executing the plan...

FINAL ANSWER:
The total budget for a 7-day trip from Riyadh to Istanbul is approximately

## Reflection Questions

1. What was the most challenging part when implementing the two agents?
2. How did you pass information from the Planner to the Executor?
3. What are the advantages of separating planning from execution?
4. What could go wrong if the Planner gives a poor plan?
5. How would you extend this to a 3-agent system (Planner + Researcher + Calculator)?
